# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
!pip -q install datasets huggingface_hub duckdb pandas pyarrow

In [20]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
login(HF_TOKEN)

In [21]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [22]:
con.sql(f"""
SELECT COUNT(*)
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [23]:
con.sql(f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
LIMIT 5
""")

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

In [24]:
df = con.sql(f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
LIMIT 100
""").df()

df.columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events',
 'month']

In [25]:
con.sql(f"""
SELECT
    MIN(month) AS first_month,
    MAX(month) AS last_month,
    COUNT(DISTINCT month) AS total_months
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

┌─────────────┬────────────┬──────────────┐
│ first_month │ last_month │ total_months │
│   varchar   │  varchar   │    int64     │
├─────────────┼────────────┼──────────────┤
│ 2025-01     │ 2026-06    │           18 │
└─────────────┴────────────┴──────────────┘

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis

One row represents the performance of one content page for one client on one reporting date.

The warehouse contains data from January 2025 to June 2026 (18 months). For development, I will use a mid-period month such as March 2026.

In [26]:
con.sql(f"""
SELECT
    MIN(month) AS first_month,
    MAX(month) AS last_month,
    COUNT(DISTINCT month) AS total_months
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

┌─────────────┬────────────┬──────────────┐
│ first_month │ last_month │ total_months │
│   varchar   │  varchar   │    int64     │
├─────────────┼────────────┼──────────────┤
│ 2025-01     │ 2026-06    │           18 │
└─────────────┴────────────┴──────────────┘

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id || '-' || content_hash_id || '-' || CAST(report_date AS VARCHAR)) AS unique_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Category     | Fields                                                                                                                                   | Reason                                              |
| ------------ | ---------------------------------------------------------------------------------------------------------------------------------------- | --------------------------------------------------- |
| **Features** | `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, `scroll_events`                                                     | Ye prediction ke inputs ho sakte hain.              |
| **Label**    | *Notebook/assignment ke target ke hisab se (agar baad me define hota hai)*                                                               | Ye woh value hogi jise predict karna hai.           |
| **Context**  | `client_hash_id`, `content_hash_id`, `report_date`, `month`, `client_has_gsc`, `client_has_ga4`                                          | Identification aur filtering ke liye useful.        |
| **Excluded** | `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` *|


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

In [ ]:
con.sql(f"""
SELECT
MIN(month) AS first_month,
MAX(month) AS last_month
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

In [ ]:
con.sql(f"""
SELECT
SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS missing_impressions,
SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS missing_clicks,
SUM(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END) AS missing_sessions
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

In [ ]:
con.sql(f"""
SELECT
gsc_data_available,
ga4_data_available,
COUNT(*) AS rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
GROUP BY gsc_data_available, ga4_data_available
ORDER BY rows DESC
""")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

- The warehouse contains data from January 2025 to June 2026 only.
- Early records may have Google Search Console data without complete GA4 metrics.
- Client and content identifiers are hashed, so the original URLs and client names cannot be inspected.
- This dataset measures observed traffic and engagement metrics. It does not directly measure content quality or business outcomes.
- The data should be used for decision support rather than causal conclusions.

## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.